# core

> Paths for weights, shipped assets, and writable app data.


In [ ]:
#| default_exp core


In [ ]:
#| hide
from nbdev.showdoc import *


In [ ]:
#| export
from __future__ import annotations


In [ ]:
#| export
import os
from pathlib import Path


In [ ]:
#| export
def package_dir() -> Path:
    """Directory of the installed `nadakit` package."""
    return Path(__file__).resolve().parent


In [ ]:
#| export
def models_dir() -> Path:
    """Directory holding `wav2vec2.safetensors` and the HF vocab snapshot.

    1. `NADA_MODELS` if set
    2. else `~/.cache/nadakit/models`
    """
    env = os.environ.get("NADA_MODELS")
    if env:
        return Path(env).expanduser()
    return Path.home() / ".cache" / "nadakit" / "models"


In [ ]:
#| export
def embed_center_path() -> Path:
    """`embed_center.npy`: under `models_dir()` first, then the copy shipped with the package."""
    local = models_dir() / "embed_center.npy"
    if local.is_file():
        return local
    shipped = package_dir() / "embed_center.npy"
    if shipped.is_file():
        return shipped
    return local


In [ ]:
#| export
def checkout_root() -> Path | None:
    """Repo checkout when developing from source, else `None`."""
    starts = [package_dir(), Path.cwd()]
    for start in starts:
        for parent in (start, *start.parents):
            toml = parent / "pyproject.toml"
            if not ((parent / "pixi.toml").is_file() and toml.is_file()):
                continue
            if 'name = "nadakit"' in toml.read_text():
                return parent
    return None


In [ ]:
#| export
def asset_dir() -> Path:
    """`glyph_table.json`, `static/`, `frontend/` — package dir or checkout."""
    pkg = package_dir()
    if (pkg / "glyph_table.json").is_file():
        return pkg
    root = checkout_root()
    if root and (root / "glyph_table.json").is_file():
        return root
    return pkg


In [ ]:
#| export
def data_dir() -> Path:
    """Writable app data (SQLite sound library)."""
    env = os.environ.get("NADAKIT_DATA")
    if env:
        path = Path(env).expanduser()
        path.mkdir(parents=True, exist_ok=True)
        return path
    root = checkout_root()
    if root:
        path = root / "data"
        path.mkdir(exist_ok=True)
        return path
    path = Path.home() / ".local" / "share" / "nadakit"
    path.mkdir(parents=True, exist_ok=True)
    return path
